In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import PorterStemmer
from collections import Counter

# Working with Text Lab
## Information retrieval, preprocessing, and feature extraction

In this lab, you'll be looking at and exploring European restaurant reviews. The dataset is rather tiny, but that's just because it has to run on any machine. In real life, just like with images, texts can be several terabytes long.

The dataset is located [here](https://www.kaggle.com/datasets/gorororororo23/european-restaurant-reviews) and as always, it's been provided to you in the `data/` folder.

### Problem 1. Read the dataset (1 point)
Read the dataset, get acquainted with it. Ensure the data is valid before you proceed.

How many observations are there? Which country is the most represented? What time range does the dataset represent?

Is the sample balanced in terms of restaurants, i.e., do you have an equal number of reviews for each one? Most importantly, is the dataset balanced in terms of **sentiment**?

In [2]:
reviews = pd.read_csv("data/European Restaurant Reviews.csv")
reviews

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,May 2024 •,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,Feb 2024 •,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,Nov 2023 •,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,Mar 2023 •,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,Nov 2022 •,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,Oct 2016 •,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,Oct 2016 •,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,Oct 2016 •,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,Oct 2016 •,We visited the Havana's Club Museum which is l...


In [3]:
reviews.dtypes

Country            object
Restaurant Name    object
Sentiment          object
Review Title       object
Review Date        object
Review             object
dtype: object

We clean Review Date column: replace 'Sept' with 'Sep', remove '•'. Then, convert to datetime (parse Month Year).

In [4]:
reviews['Review Date'] = reviews['Review Date'].str.replace('Sept', 'Sep').str.replace('•','').str.strip()

In [5]:
reviews['Review Date'] = pd.to_datetime(reviews['Review Date'], format='%b %Y')

In [6]:
reviews

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,2024-05-01,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,2024-02-01,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,2023-11-01,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,2023-03-01,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,2022-11-01,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,2016-10-01,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,2016-10-01,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,2016-10-01,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,2016-10-01,We visited the Havana's Club Museum which is l...


In [7]:
reviews.dtypes

Country                    object
Restaurant Name            object
Sentiment                  object
Review Title               object
Review Date        datetime64[ns]
Review                     object
dtype: object

In [8]:
num_observations = len(reviews)

In [9]:
most_country = reviews['Country'].value_counts().idxmax()

In [10]:
date_min = reviews['Review Date'].min()
date_max = reviews['Review Date'].max()

In [11]:
restaurant_counts = reviews['Restaurant Name'].value_counts()

In [12]:
sentiment_counts = reviews['Sentiment'].value_counts()

In [13]:
print(f"Total reviews: {num_observations}")
print(f"Most reviews from: {most_country}")
print(f"Date range: {date_min.strftime('%b %Y')} to {date_max.strftime('%b %Y')}\n")

Total reviews: 1502
Most reviews from: France
Date range: Sep 2010 to Jul 2024



In [14]:
print("Reviews per restaurant:")
for name, count in restaurant_counts.items():
    print(f"  {name}: {count}")

Reviews per restaurant:
  The Frog at Bercy Village: 512
  Ad Hoc Ristorante (Piazza del Popolo): 318
  The LOFT: 210
  Old Square (Plaza Vieja): 146
  Stara Kamienica: 135
  Pelmenya: 100
  Mosaic: 81


For restaurant balance: no, not at all - review counts range from 81 (Mosaic) up to 512 (The Frog at Bercy Village), so some restaurants are heavily over‑represented while others have relatively few reviews.

In [15]:
print("\nSentiment distribution:")
for sentiment, count in sentiment_counts.items():
    print(f"  {sentiment}: {count}")


Sentiment distribution:
  Positive: 1237
  Negative: 265


For sentiment balance: also not balanced at all - there are 1237 positive vs only 265 negative reviews, so roughly 80% positive and 20% negative.

### Problem 2. Getting acquainted with reviews (1 point)
Are positive comments typically shorter or longer? Try to define a good, robust metric for "length" of a text; it's not necessary just the character count. Can you explain your findings?

In [16]:
# We define length metrics
reviews['word_count'] = reviews['Review'].str.split().apply(len)

In [17]:
length_stats = reviews.groupby('Sentiment')['word_count'] \
                 .agg(['mean', 'median', 'std', 'count']) \
                 .reset_index()
length_stats

,Sentiment,mean,median,std,count
0,Negative,140.573585,95.0,131.759636,265
1,Positive,50.183508,37.0,38.741043,1237


Here are our observations: 

Negative reviews (265 total):

Mean: $\sim$ 140 words

Median: 95 words

Std Dev: $\sim$ 132 words

Positive reviews (1237 total):

Mean: $\sim$  50 words

Median: 37 words

Std Dev: $\sim$ 39 words

I used word count (number of words per review) as a robust length metric rather than simple character count, since it better reflects how much content a review contains.

My findings are connected with the fact that negative comments are substantially longer than positive ones (mean of 140 vs. 50 words). This makes sense because complaints typically require more explanation, examples and detail, whereas positive feedback is often briefer ("Great service!", "Loved it!"). The higher standard deviation for negatives also shows that some negative reviews get quite wordy. This demonstrates that sentiment correlates with review length: longer reviews tend to be negative in this dataset. 

### Problem 3. Preprocess the review content (2 points)
You'll likely need to do this while working on the problems below, but try to synthesize (and document!) your preprocessing here. Your tasks will revolve around words and their connection to sentiment. While preprocessing, keep in mind the domain (restaurant reviews) and the task (sentiment analysis).

We start with: 

1. Text Normalization - lowercasing: we need to convert all characters to lowercase to reduce vocabulary size (Great -> great). Next, we remove HTML artifacts and special symbols: strip tags and uncommon unicode (&amp;, emojis) that don't contribute to sentiment.

In [18]:
def normalize_text(text):
    """
    Function to normalize a given text - remove artifacts, symbols, convert to lowercase
    """
    
    text = text.lower()
    text = re.sub(r"&amp;|&lt;|&gt;", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)    # remove punctuation and special chars
    return text

In [19]:
reviews['CleanReview'] = reviews['Review'].apply(normalize_text)
reviews['CleanReview']

0       the manager became agressive when i said the c...
1       i ordered a beef fillet ask to be done medium ...
2       this is an attractive venue with welcoming  al...
3       sadly i  used the high tripadvisor rating too ...
4       from the start this meal was bad  especially g...
                              ...                        
1497    despite the other reviews saying that this is ...
1498    beer is good   food is awfull  the only decent...
1499    for terrible service of a truly comedic level ...
1500    we visited the havana s club museum which is l...
1501    food and service was awful  very pretty stop  ...
Name: CleanReview, Length: 1502, dtype: object

2. Tokenization - whitespace tokenization - the simplest and fastest approach is to split on whitespace. This works well for our cleaned text.

In [20]:
reviews['Tokens'] = reviews['CleanReview'].str.split()
reviews['Tokens']

0       [the, manager, became, agressive, when, i, sai...
1       [i, ordered, a, beef, fillet, ask, to, be, don...
2       [this, is, an, attractive, venue, with, welcom...
3       [sadly, i, used, the, high, tripadvisor, ratin...
4       [from, the, start, this, meal, was, bad, espec...
                              ...                        
1497    [despite, the, other, reviews, saying, that, t...
1498    [beer, is, good, food, is, awfull, the, only, ...
1499    [for, terrible, service, of, a, truly, comedic...
1500    [we, visited, the, havana, s, club, museum, wh...
1501    [food, and, service, was, awful, very, pretty,...
Name: Tokens, Length: 1502, dtype: object

3. Stopword Removal - instead of manual lists, we will use scikit-learn's built-in removal in one step during feature extraction.

In [21]:
vectorizer = TfidfVectorizer(
    stop_words='english',  
    tokenizer= lambda text: text.split(),
    lowercase=False,
    token_pattern = None
)

X_tfidf = vectorizer.fit_transform(reviews['CleanReview'])
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 42447 stored elements and shape (1502, 6032)>

4. Stemming - to avoid external model dependencies, we will apply Porter stemming using NLTK's built-in stemmer.

In [22]:
stemmer = PorterStemmer()

In [23]:
def stem_tokens(tokens):
    """
    Function to stem received tokens
    """
    
    return [stemmer.stem(t) for t in tokens]

In [24]:
reviews['Stems'] = reviews['Tokens'].apply(stem_tokens)
reviews['Stems']

0       [the, manag, becam, agress, when, i, said, the...
1       [i, order, a, beef, fillet, ask, to, be, done,...
2       [thi, is, an, attract, venu, with, welcom, alb...
3       [sadli, i, use, the, high, tripadvisor, rate, ...
4       [from, the, start, thi, meal, wa, bad, especi,...
                              ...                        
1497    [despit, the, other, review, say, that, thi, i...
1498    [beer, is, good, food, is, awful, the, onli, d...
1499    [for, terribl, servic, of, a, truli, comed, le...
1500    [we, visit, the, havana, s, club, museum, whic...
1501    [food, and, servic, wa, aw, veri, pretti, stop...
Name: Stems, Length: 1502, dtype: object

5. Handling Negation - we will apply only if we need to capture flipped sentiment ("not good").

In [25]:
def handle_negation(tokens):
    """
    Function to handle the negation of words with received tokens
    """
    
    out = []
    neg = False
    for t in tokens:
        if t in {'not', 'no', 'never'}:
            neg = True
            continue
        if neg:
            out.append(f"not_{t}")
            neg = False
        else:
            out.append(t)
    return out

In [26]:
reviews['NegTokens'] = reviews['Stems'].apply(handle_negation)
reviews['NegTokens']

0       [the, manag, becam, agress, when, i, said, the...
1       [i, order, a, beef, fillet, ask, to, be, done,...
2       [thi, is, an, attract, venu, with, welcom, alb...
3       [sadli, i, use, the, high, tripadvisor, rate, ...
4       [from, the, start, thi, meal, wa, bad, especi,...
                              ...                        
1497    [despit, the, other, review, say, that, thi, i...
1498    [beer, is, good, food, is, awful, the, onli, d...
1499    [for, terribl, servic, of, a, truli, comed, le...
1500    [we, visit, the, havana, s, club, museum, whic...
1501    [food, and, servic, wa, aw, veri, pretti, stop...
Name: NegTokens, Length: 1502, dtype: object

### Problem 4. Top words (1 point)
Use a simple word tokenization and count the top 10 words in positive reviews; then the top 10 words in negative reviews*. Once again, try to define what "top" words means. Describe and document your process. Explain your results.

\* Okay, you may want to see top N words (with $N \ge 10$).

We define "top" words as those with the highest term frequency (raw count) within each sentiment subset.
We exclude any tokens of length $\le$ 2 (like "it", "in") to focus on more meaningful words.

In [27]:
def filter_tokens(tok_list):
    """
    Function to filter out short tokens, TF-IDF stopwords, and domain words
    """
    
    return [
        t for t in tok_list
        if len(t) > 2
        and t not in tfidf_stop
        and t not in domain_stop
    ]

In [28]:
# Build stopword sets
tfidf_stop = set(vectorizer.get_stop_words())
domain_stop = {'food', 'service', 'restaurant', 'menu', 'place', 'wine', 'staff'}

# Gather tokens by sentiment
positive_tokens = reviews.loc[reviews['Sentiment'] == 'Positive', 'Tokens'].sum()
negative_tokens = reviews.loc[reviews['Sentiment'] == 'Negative', 'Tokens'].sum()

positive_filtered = filter_tokens(positive_tokens)
negative_filtered = filter_tokens(negative_tokens)

# Count frequencies
positive_counts = Counter(positive_filtered)
negative_counts = Counter(negative_filtered)

# Compute difference and get top N
diff = {}
all_terms = set(positive_counts) | set(negative_counts)
for term in all_terms:
    diff[term] = positive_counts.get(term, 0) - negative_counts.get(term, 0)

top_N = 10

# Top N distinctly positive (largest positive diff)
top_pos = sorted(diff.items(), key=lambda x: -x[1])[:top_N]

# Top N distinctly negative (most negative diff)
top_neg = sorted(diff.items(), key=lambda x: x[1])[:top_N]

print("Top 10 Distinctly Positive Words:")
for w, d in top_pos:
    print(f"  {w:15} (+{d})")

print("\nTop 10 Distinctly Negative Words:")
for w, d in top_neg:
    print(f"  {w:15} ({d})")

Top 10 Distinctly Positive Words:
  great           (+520)
  good            (+361)
  excellent       (+221)
  delicious       (+221)
  nice            (+220)
  friendly        (+218)
  amazing         (+189)
  recommend       (+171)
  atmosphere      (+161)
  lovely          (+154)

Top 10 Distinctly Negative Words:
  table           (-77)
  asked           (-75)
  minutes         (-61)
  average         (-53)
  cold            (-46)
  told            (-46)
  said            (-41)
  reviews         (-40)
  took            (-38)
  did             (-37)


These "distinctly" lists look solid. They show the words most over‑represented in each class after filtering out generic terms. Distinctly Positive: great, good, excellent, delicious, etc., are exactly the praise words we'd expect, each with a large positive difference (like 520 more occurrences in positives than negatives). Distinctly Negative: table, asked, minutes, average, cold, etc., reflect typical complaints about waits, seating, or subpar food.

### Problem 5. Review titles (2 point)
How do the top words you found in the last problem correlate to the review titles? Do the top 10 words (for each sentiment) appear in the titles at all? Do reviews which contain one or more of the top words have the same words in their titles?

Does the title of a comment present a good summary of its content? That is, are the titles descriptive, or are they simply meant to catch the attention of the reader?

### Problem 6. Bag of words (1 point)
Based on your findings so far, come up with a good set of settings (hyperparameters) for a bag-of-words model for review titles and contents. It's easiest to treat them separately (so, create two models); but you may also think about a unified representation. I find the simplest way of concatenating the title and content too simplistic to be useful, as it doesn't allow you to treat the title differently (e.g., by giving it more weight).

The documentation for `CountVectorizer` is [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html). Familiarize yourself with all settings; try out different combinations and come up with a final model; or rather - two models :).

### Problem 7. Deep sentiment analysis models (1 point)
Find a suitable model for sentiment analysis in English. Without modifying, training, or fine-tuning the model, make it predict all contents (or better, combinations of titles and contents, if you can). Meaure the accuracy of the model compared to the `sentiment` column in the dataset.

### Problem 8. Deep features (embeddings) (1 point)
Use the same model to perform feature extraction on the review contents (or contents + titles) instead of direct predictions. You should already be familiar how to do that from your work on images.

Use the cosine similarity between texts to try to cluster them. Are there "similar" reviews (you'll need to find a way to measure similarity) across different restaurants? Are customers generally in agreement for the same restaurant?

### \* Problem 9. Explore and model at will
In this lab, we focused on preprocessing and feature extraction and we didn't really have a chance to train (or compare) models. The dataset is maybe too small to be conclusive, but feel free to play around with ready-made models, and train your own.